# YouTube Clipper - Google Colab

Transforms long-form YouTube videos into vertical shorts (9:16, 1080×1920).

**Features:**
- AI-suggested highlight clips from transcript
- TikTok-style word-by-word captions
- GPU-accelerated encoding (when available)
- Persistent output to Google Drive

**Setup:**
1. Go to `Runtime > Change runtime type > GPU`
2. Add your API keys to Colab Secrets (🔑 icon in left sidebar):
   - `OPENROUTER_API_KEY`
   - `SUPADATA_API_KEY` (optional, for transcript fetching)

## 1. Setup - Install Dependencies

In [8]:
# Install system dependencies
!apt-get update -qq && apt-get install -y -qq ffmpeg > /dev/null 2>&1

# Remove existing dir if it exists to ensure a clean clone
!rm -rf /content/yt-clipping

# Clone the yt-clipping repo
!git clone https://github.com/techafreshh/yt-clipping.git /content/yt-clipping

# Install Python dependencies
!pip install -q yt-dlp bgutil-ytdlp-pot-provider httpx pydantic pydantic-settings python-dotenv fastapi uvicorn python-multipart

# Attempt installation - if editable fails, we still have the folder for sys.path
!pip install -q /content/yt-clipping || echo 'Notice: standard install failed, using manual path injection'

print("✅ Setup complete")

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Cloning into '/content/yt-clipping'...
remote: Enumerating objects: 116, done.
remote: Counting objects: 100% (116/116), done.
remote: Compressing objects: 100% (83/83), done.
remote: Total 116 (delta 23), reused 114 (delta 21), pack-reused 0 (from 0)
Receiving objects: 100% (116/116), 144.06 KiB | 957.00 KiB/s, done.
Resolving deltas: 100% (23/23), done.
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 149.4/149.4 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 94.8/94.8 kB 11.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 76.4/76.4 kB 9.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 434.9/434.9 kB 24.5 MB/s eta 0:00:00

## 2. Mount Google Drive

In [2]:
from google.colab import drive
drive.mount('/content/drive')

# Create base shorts directory on Drive
import os
DRIVE_SHORTS_DIR = "/content/drive/MyDrive/shorts"
os.makedirs(DRIVE_SHORTS_DIR, exist_ok=True)
print(f"✅ Drive mounted. Output will be saved to: {DRIVE_SHORTS_DIR}")

Mounted at /content/drive
✅ Drive mounted. Output will be saved to: /content/drive/MyDrive/shorts


## 3. Configure API Keys

In [3]:
from google.colab import userdata
import os

# Load API keys from Colab Secrets
try:
    os.environ["OPENROUTER_API_KEY"] = userdata.get("OPENROUTER_API_KEY")
    print("✅ OPENROUTER_API_KEY loaded")
except:
    print("⚠️ OPENROUTER_API_KEY not found in Colab Secrets")
    print("   Add it via the 🔑 icon in the left sidebar")

try:
    os.environ["SUPADATA_API_KEY"] = userdata.get("SUPADATA_API_KEY")
    print("✅ SUPADATA_API_KEY loaded")
except:
    print("ℹ️ SUPADATA_API_KEY not found (optional - will use yt-dlp fallback)")

✅ OPENROUTER_API_KEY loaded
✅ SUPADATA_API_KEY loaded


## 4. GPU Detection & Helper Functions

In [9]:
import subprocess
import sys
import shutil
from pathlib import Path

# Check GPU availability
def check_gpu():
    """Check if GPU is available and NVENC is supported."""
    try:
        result = subprocess.run(["nvidia-smi"], capture_output=True, text=True, timeout=5)
        if result.returncode == 0:
            print("✅ GPU available:")
            # Extract GPU name from nvidia-smi output
            for line in result.stdout.split("\n"):
                if "NVIDIA" in line:
                    print(f"   {line.strip()}")
                    break

            # Check NVENC support
            encoders = subprocess.run(["ffmpeg", "-encoders"], capture_output=True, text=True)
            if "h264_nvenc" in encoders.stdout:
                print("   NVENC encoding: ✅ Supported")
            else:
                print("   NVENC encoding: ❌ Not supported (will use CPU)")
            return True
    except:
        pass

    print("ℹ️ No GPU detected - will use CPU encoding")
    return False

check_gpu()

# Add shorts package to path
sys.path.insert(0, "/content/yt-clipping/src")
os.chdir("/content/yt-clipping")

print("\n✅ Helper functions loaded")

# Interactive crop selection inside Colab cell
def select_crop_colab(video_name: str, clip_idx: int = 0):
    """
    Renders an interactive 9:16 crop selector inside the Google Colab cell.
    Extracts a frame from the video, displays it, allows dragging a 9:16 box,
    and updates the clip's crop in clips.json.
    """
    from IPython.display import display, HTML
    from google.colab import output
    import json
    import base64
    import subprocess
    from pathlib import Path
    from shorts.highlights import load_clips, save_clips, parse_timestamp
    
    clips = load_clips(video_name)
    if not clips or clip_idx >= len(clips):
        print(f"Error: Clip index {clip_idx} not found (total clips: {len(clips) if clips else 0}).")
        return
        
    clip = clips[clip_idx]
    start_sec = parse_timestamp(clip.start)
    mid_sec = start_sec + 2.0
    
    video_path = Path("raw") / f"{video_name}.mp4"
    frame_path = Path("working") / video_name / f"{clip.slug}_frame.jpg"
    frame_path.parent.mkdir(parents=True, exist_ok=True)
    
    # Extract frame via ffmpeg
    subprocess.run([
        "ffmpeg", "-y", "-ss", str(mid_sec), "-i", str(video_path),
        "-frames:v", "1", "-q:v", "2", str(frame_path)
    ], capture_output=True)
    if not frame_path.exists():
        subprocess.run([
            "ffmpeg", "-y", "-ss", str(start_sec), "-i", str(video_path),
            "-frames:v", "1", "-q:v", "2", str(frame_path)
        ], capture_output=True)
        
    if not frame_path.exists():
        print(f"Error: Could not extract frame from {video_path}")
        return

    with open(frame_path, "rb") as f:
        img_base64 = base64.b64encode(f.read()).decode("utf-8")
        
    def save_crop_callback(x, y, w, h):
        local_clips = load_clips(video_name)
        from shorts.highlights import CropRegion
        local_clips[clip_idx].crop = CropRegion(x=x, y=y, w=w, h=h)
        save_clips(video_name, local_clips)
        print(f"✅ Applied and saved 9:16 crop for '{clip.slug}'!")
        
    output.register_callback('notebook_save_crop', save_crop_callback)
    
    html_content = f"""
    <div style="font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, sans-serif; max-width: 800px; margin: auto; padding: 15px; background: #1e1e2e; color: #cdd6f4; border-radius: 12px; border: 1px solid #313244;">
        <h3 style="margin-top: 0; color: #cba6f7; display: flex; align-items: center; gap: 8px;">
            <svg width="24" height="24" viewBox="0 0 24 24" fill="none" stroke="currentColor" stroke-width="2" stroke-linecap="round" stroke-linejoin="round"><path d="M6 2v14a2 2 0 0 0 2 2h14"></path><path d="M18 22V8a2 2 0 0 0-2-2H2"></path></svg>
            Interactive 9:16 Crop Selection: <span style="font-family: monospace; background: #313244; padding: 2px 8px; border-radius: 4px; font-size: 0.95em;">{clip.slug}</span>
        </h3>
        <p style="font-size: 0.9rem; color: #a6adc8; margin-bottom: 12px;">
            Click and drag on the image to select the vertical crop focus region (locked to 9:16 aspect ratio).
        </p>
        <div style="position: relative; display: inline-block; border: 2px solid #45475a; border-radius: 6px; overflow: hidden; background: #11111b;">
            <img id="crop-source-img" src="data:image/jpeg;base64,{img_base64}" style="max-width: 100%; max-height: 500px; display: block;" />
            <canvas id="crop-canvas" style="position: absolute; top: 0; left: 0; cursor: crosshair;"></canvas>
        </div>
        <div style="margin-top: 15px; display: flex; gap: 12px; align-items: center;">
            <button id="save-crop-btn" style="background: #a6e3a1; color: #11111b; border: none; padding: 10px 20px; font-weight: bold; border-radius: 6px; cursor: pointer; transition: background 0.2s;">Apply Crop</button>
            <button id="reset-crop-btn" style="background: #f38ba8; color: #11111b; border: none; padding: 10px 20px; font-weight: bold; border-radius: 6px; cursor: pointer; transition: background 0.2s;">Reset Selection</button>
            <span id="crop-status-text" style="color: #bac2de; font-size: 0.9rem; font-family: monospace;">Calculating defaults...</span>
        </div>
    </div>
    <script>
    (function() {{
        const img = document.getElementById('crop-source-img');
        const canvas = document.getElementById('crop-canvas');
        const ctx = canvas.getContext('2d');
        const status = document.getElementById('crop-status-text');
        
        let imgWidth, imgHeight;
        let isDrawing = false;
        let startX, startY;
        let cropX = 0.375, cropY = 0, cropW = 0.25, cropH = 1;
        
        function init() {{
            imgWidth = img.width || img.clientWidth;
            imgHeight = img.height || img.clientHeight;
            canvas.width = imgWidth;
            canvas.height = imgHeight;
            
            const imgAspect = imgWidth / imgHeight;
            const targetAspect = 9 / 16;
            
            if (imgAspect > targetAspect) {{
                cropH = 1.0;
                cropW = targetAspect / imgAspect;
                cropX = (1.0 - cropW) / 2;
                cropY = 0;
            }} else {{
                cropW = 1.0;
                cropH = imgAspect / targetAspect;
                cropX = 0;
                cropY = (1.0 - cropH) / 2;
            }}
            drawCrop();
        }}
        
        img.onload = init;
        if (img.complete) init();
        
        function drawCrop() {{
            ctx.clearRect(0, 0, canvas.width, canvas.height);
            ctx.fillStyle = 'rgba(0, 0, 0, 0.6)';
            ctx.fillRect(0, 0, canvas.width, canvas.height);
            
            const cx = cropX * canvas.width;
            const cy = cropY * canvas.height;
            const cw = cropW * canvas.width;
            const ch = cropH * canvas.height;
            ctx.clearRect(cx, cy, cw, ch);
            
            ctx.strokeStyle = '#cba6f7';
            ctx.lineWidth = 3;
            ctx.strokeRect(cx, cy, cw, ch);
            
            ctx.strokeStyle = 'rgba(203, 166, 247, 0.3)';
            ctx.lineWidth = 1;
            ctx.beginPath();
            ctx.moveTo(cx + cw/3, cy); ctx.lineTo(cx + cw/3, cy + ch);
            ctx.moveTo(cx + 2*cw/3, cy); ctx.lineTo(cx + 2*cw/3, cy + ch);
            ctx.moveTo(cx, cy + ch/3); ctx.lineTo(cx + cw, cy + ch/3);
            ctx.moveTo(cx, cy + 2*ch/3); ctx.lineTo(cx + cw, cy + 2*ch/3);
            ctx.stroke();
            
            status.innerText = `Crop: x=${{cropX.toFixed(3)}}, y=${{cropY.toFixed(3)}}, w=${{cropW.toFixed(3)}}, h=${{cropH.toFixed(3)}}`;
        }}
        
        canvas.addEventListener('mousedown', function(e) {{
            const rect = canvas.getBoundingClientRect();
            startX = e.clientX - rect.left;
            startY = e.clientY - rect.top;
            isDrawing = true;
        }});
        
        canvas.addEventListener('mousemove', function(e) {{
            if (!isDrawing) return;
            const rect = canvas.getBoundingClientRect();
            const currentX = e.clientX - rect.left;
            const currentY = e.clientY - rect.top;
            
            let w = currentX - startX;
            let h = w * (16 / 9);
            
            let endX = startX + w;
            let endY = startY + h;
            
            if (endX < 0) endX = 0;
            if (endX > canvas.width) endX = canvas.width;
            if (endY < 0) endY = 0;
            if (endY > canvas.height) endY = canvas.height;
            
            w = endX - startX;
            h = w * (16 / 9);
            
            const normX = Math.min(startX, startX + w) / canvas.width;
            const normY = Math.min(startY, startY + h) / canvas.height;
            const normW = Math.abs(w) / canvas.width;
            const normH = Math.abs(h) / canvas.height;
            
            if (normW > 0.02 && normH > 0.02 && (normX + normW <= 1.0) && (normY + normH <= 1.0)) {{
                cropX = normX;
                cropY = normY;
                cropW = normW;
                cropH = normH;
                drawCrop();
            }}
        }});
        
        canvas.addEventListener('mouseup', function() {{
            isDrawing = false;
        }});
        
        document.getElementById('save-crop-btn').addEventListener('click', function() {{
            google.colab.kernel.invokeFunction('notebook_save_crop', [cropX, cropY, cropW, cropH], {{}});
        }});
        
        document.getElementById('reset-crop-btn').addEventListener('click', function() {{
            init();
        }});
    }})();
    </script>
    """
    display(HTML(html_content))


✅ GPU available:
   | NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
   NVENC encoding: ✅ Supported

✅ Helper functions loaded


## 5. Input - Video Configuration

In [10]:
#@title YouTube Video Configuration
#@markdown Enter the YouTube URL and clip settings below:

YOUTUBE_URL = "https://www.youtube.com/watch?v=jrAYeKTHJ3M" #@param {type:"string"}
NUM_CLIPS = 8 #@param {type:"integer"}
ADD_CAPTIONS = True #@param {type:"boolean"}
MODEL = "xiaomi/mimo-v2.5-pro" # @param ["anthropic/claude-sonnet-4","xiaomi/mimo-v2.5-pro","openai/gpt-4o"]

#@markdown Caption styling options:
CAPTION_ALIGNMENT = 2 #@param {type:"integer"}
CAPTION_MARGIN_V = 480 #@param {type:"integer"}

#@markdown Webhook notifications:
N8N_WEBHOOK_URL = "" #@param {type:"string"}

# Derive video name using yt-dlp derived title if available
import re
import sys
sys.path.insert(0, "/content/yt-clipping/src")
from shorts.downloader import get_youtube_title, derive_name_from_url

try:
    title = get_youtube_title(YOUTUBE_URL)
    name = title or derive_name_from_url(YOUTUBE_URL)
except Exception:
    name = derive_name_from_url(YOUTUBE_URL)

VIDEO_NAME = re.sub(r"[^a-zA-Z0-9_-]", "_", name).strip("_")

print(f"Video name (folder name): {VIDEO_NAME}")
print(f"Clips to generate: {NUM_CLIPS}")
print(f"Captions: {'Enabled' if ADD_CAPTIONS else 'Disabled'}")
print(f"Model: {MODEL}")

Video name: jrayekthj3m
Clips to generate: 8
Captions: Enabled
Model: xiaomi/mimo-v2.5-pro


## 6. Run Pipeline

In [11]:
import os
import sys
import json
import shutil
from pathlib import Path

# Add shorts package to path first
sys.path.insert(0, "/content/yt-clipping/src")

# Configure shorts settings dynamically from user parameters
from shorts.config import settings
settings.caption_alignment = CAPTION_ALIGNMENT
settings.caption_margin_v = CAPTION_MARGIN_V
if N8N_WEBHOOK_URL:
    settings.n8n_webhook_url = N8N_WEBHOOK_URL

# Set up per-video directory structure
video_dir = Path(DRIVE_SHORTS_DIR) / VIDEO_NAME
video_dir.mkdir(parents=True, exist_ok=True)

# Create subdirectories
(video_dir / "raw").mkdir(exist_ok=True)
(video_dir / "transcripts").mkdir(exist_ok=True)
(video_dir / "clips").mkdir(exist_ok=True)
(video_dir / "output").mkdir(exist_ok=True)

# Working directory (local for speed, cleaned up after)
working_dir = Path("/content/working") / VIDEO_NAME
working_dir.mkdir(parents=True, exist_ok=True)

# Create symlinks so shorts package finds files in the right places
for subdir in ["raw", "transcripts", "clips", "output"]:
    local_path = Path(subdir)
    drive_path = video_dir / subdir
    if local_path.exists() and not local_path.is_symlink():
        shutil.rmtree(local_path)
    if not local_path.exists():
        local_path.symlink_to(drive_path)

# Working directory symlink
local_working = Path("working")
if local_working.exists() and not local_working.is_symlink():
    shutil.rmtree(local_working)
if not local_working.exists():
    local_working.symlink_to(working_dir)

print(f"📁 Video directory: {video_dir}")
print(f"📁 Working directory: {working_dir}")

# Change to the yt-clipping directory
os.chdir("/content/yt-clipping")

📁 Video directory: /content/drive/MyDrive/shorts/jrayekthj3m
📁 Working directory: /content/working/jrayekthj3m


In [12]:
# Step 1: Download video
print("📥 Step 1: Downloading video...")
from shorts.downloader import download_youtube

try:
    download_youtube(YOUTUBE_URL, VIDEO_NAME)
    print(f"   ✅ Downloaded to raw/{VIDEO_NAME}.mp4")
except Exception as e:
    print(f"   ❌ Download failed: {e}")
    raise

📥 Step 1: Downloading video...
   ✅ Downloaded to raw/jrayekthj3m.mp4


In [13]:
# Step 2: Fetch transcript
print("📝 Step 2: Fetching transcript...")
from shorts.transcript import fetch_transcript_with_fallback, save_transcript
from shorts.config import settings

api_key = getattr(settings, "supadata_api_key", None)
try:
    transcript = fetch_transcript_with_fallback(YOUTUBE_URL, VIDEO_NAME, api_key)
    save_transcript(VIDEO_NAME, transcript)
    print(f"   ✅ Transcript saved ({len(transcript.segments)} segments)")
except Exception as e:
    print(f"   ❌ Transcript failed: {e}")
    raise

📝 Step 2: Fetching transcript...
   ✅ Transcript saved (291 segments)


In [14]:
# Step 3: AI suggest highlights
print("🤖 Step 3: AI suggesting highlights...")
from shorts.config import require
from shorts.highlights import suggest_highlights, validate_clips, save_clips
from shorts.transcript import load_cached

cached = load_cached(VIDEO_NAME)
api_key = require(settings, "openrouter_api_key")
max_duration = max(seg.end for seg in cached.segments)

try:
    clips = suggest_highlights("", api_key, MODEL, NUM_CLIPS, segments=cached.segments, total_duration=max_duration)
    clips = validate_clips(clips, max_duration)
    save_clips(VIDEO_NAME, clips)
    print(f"   ✅ Generated {len(clips)} clips:")
    for i, clip in enumerate(clips, 1):
        duration = float(clip.end.split(":")[-1]) - float(clip.start.split(":")[-1]) if ":" in clip.start else 0
        print(f"      {i}. {clip.slug} ({clip.start} → {clip.end})")
        if clip.hook:
            print(f"         Hook: {clip.hook}")
except Exception as e:
    print(f"   ❌ Suggestion failed: {e}")
    raise

🤖 Step 3: AI suggesting highlights...
   ✅ Generated 6 clips:
      1. never-get-back-with-ex (00:12 → 00:54)
         Hook: Your ex is an ex for a reason.
      2. never-restore-her-status (01:41 → 02:25)
         Hook: Never, ever, ever restore her to her previous level.
      3. cant-have-same-relationship-twice (03:33 → 04:16)
         Hook: You can't step into the same river twice.
      4. cant-recapture-the-feeling (04:46 → 05:25)
         Hook: You're probably trying to recapture a feeling.
      5. aggressive-ex-no-go (06:26 → 07:05)
         Hook: There are two kinds of exes you can't even be casual with.
      6. in-love-ex-casual-relationship (08:14 → 09:06)
         Hook: The ex you're still in love with.


## Optional: Interactive Crop Selection

**Note: You must run Step 3 (AI suggest highlights) above before running this cell so that the clips are generated.**

To override the default center crop, run this cell for any clip index. Draw a box on the frame, click **Apply Crop**, and then run **Step 4: Cutting and rendering** below to export the clips with your custom crops.

In [ ]:
# Change clip_idx to select crop for different clips (0-indexed)
select_crop_colab(VIDEO_NAME, clip_idx=0)

In [ ]:
import os
import shutil
from pathlib import Path
from shorts.cutter import cut_clip
from shorts.captions import generate_ass
from shorts.highlights import load_clips, parse_timestamp

print("✂️ Step 4: Cutting and rendering clips in single pass with GPU acceleration...")
clips = load_clips(VIDEO_NAME)
success = 0
errors = []

# Ensure output directory exists
final_output_base = Path("output") / VIDEO_NAME
final_output_base.mkdir(parents=True, exist_ok=True)

for i, clip in enumerate(clips, 1):
    print(f"\n   [{i}/{len(clips)}] Processing: {clip.slug}")
    try:
        # 1. Generate captions if enabled
        subtitle_path = None
        if ADD_CAPTIONS:
            print("      📝 Generating captions...")
            subtitle_path = generate_ass(
                VIDEO_NAME, clip.slug, cached,
                parse_timestamp(clip.start), parse_timestamp(clip.end)
            )

        # 2. Render Vertical Crop and Burn Subtitles (Single Pass)
        print("      🔥 Rendering clip (single pass GPU acceleration)...")
        clip_crop = clip.crop.model_dump() if clip.crop else None
        result = cut_clip(VIDEO_NAME, clip, crop=clip_crop, subtitle_path=subtitle_path)
        export_path = result.video_path

        # 3. Save to Drive
        dest_filename = f"{VIDEO_NAME}_short_{i:02d}_{clip.slug}.mp4"
        output_path = final_output_base / dest_filename

        if export_path.exists():
            shutil.copy2(export_path, output_path)
            print(f"      ✅ Exported to Drive: {dest_filename}")
            success += 1
        else:
            print(f"      ❌ Error: Source file {export_path} not found.")

    except Exception as e:
        print(f"      ❌ Failed: {str(e)}")
        errors.append(f"{clip.slug}: {str(e)}")

print(f"\n📊 Final Results: {success}/{len(clips)} clips successfully exported")
if errors:
    print("\n⚠️ Some clips failed to process.")

# 4. Trigger n8n webhook if configured
if N8N_WEBHOOK_URL:
    print("\nTriggering n8n webhook...")
    try:
        import httpx
        payload = {
            "video_name": VIDEO_NAME,
            "youtube_url": YOUTUBE_URL,
            "status": "success" if success == len(clips) else "partial_or_failure",
            "clips_total": len(clips),
            "clips_success": success,
            "clips_failed": len(errors),
            "clips": [
                {
                    "slug": c.slug,
                    "start": c.start,
                    "end": c.end,
                    "hook": getattr(c, "hook", ""),
                } for c in clips
            ],
            "errors": errors,
        }
        resp = httpx.post(N8N_WEBHOOK_URL, json=payload, timeout=15)
        print(f"   n8n webhook status: {resp.status_code}")
    except Exception as we:
        print(f"   ⚠️ Failed to trigger n8n webhook: {we}")

✂️ Step 4: Cutting clips with GPU acceleration...

   [1/6] Processing: never-get-back-with-ex
      🔥 Encoding with GPU (NVENC)...
      ✅ Exported to Drive: jrayekthj3m_short_01_never-get-back-with-ex.mp4

   [2/6] Processing: never-restore-her-status
      🔥 Encoding with GPU (NVENC)...
      ✅ Exported to Drive: jrayekthj3m_short_02_never-restore-her-status.mp4

   [3/6] Processing: cant-have-same-relationship-twice
      🔥 Encoding with GPU (NVENC)...
      ✅ Exported to Drive: jrayekthj3m_short_03_cant-have-same-relationship-twice.mp4

   [4/6] Processing: cant-recapture-the-feeling


## 7. Output - Files on Drive

In [ ]:
# List output files
output_dir = Path("output") / VIDEO_NAME
if output_dir.exists():
    files = list(output_dir.glob("*.mp4"))
    print(f"📁 Output files ({len(files)} clips):")
    for f in sorted(files):
        size_mb = f.stat().st_size / (1024 * 1024)
        print(f"   {f.name} ({size_mb:.1f} MB)")

    print(f"\n📂 Files saved to: {video_dir}/output/")
    print("   Access them in Google Drive under: My Drive/shorts/")
else:
    print("❌ No output files found")

In [ ]:
# Cleanup working directory to free space
import shutil
working_path = Path("working")
if working_path.exists() and working_path.is_symlink():
    target = working_path.resolve()
    working_path.unlink()
    if target.exists():
        shutil.rmtree(target, ignore_errors=True)
    print("🧹 Working directory cleaned up")

# Remove symlinks
for subdir in ["raw", "transcripts", "clips", "output"]:
    p = Path(subdir)
    if p.is_symlink():
        p.unlink()

print("✅ Done! Your clips are in Google Drive under: My Drive/shorts/")